In [0]:
%sql
-- views
CREATE OR REPLACE TEMP VIEW vw_chamados_hora AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/Chamados_Hora.CSV', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_base_atendentes AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/base_atendentes.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_base_motivos AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/base_motivos.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_canais AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/canais.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_chamados AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/chamados.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_clientes AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/clientes.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_custos AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/custos.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TEMP VIEW vw_pesquisa_satisfacao AS
SELECT * FROM read_files('/Volumes/rocket_lab/default/rocket_lab/case_vcredit/pesquisa_satisfacao.csv', format => 'csv', header => true, inferSchema => true);

# INSTRUÇÕES GERAIS

- padronizar tudo pra caixa baixa (inclusive cabeçalhos) e tirar acentos
- verificar número de anulados em cada tabela pra não correr o risco de anular indevidamente
- anular tudo o que não cair nos padrões definidos
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.

## CLIENTES

| Coluna | Formato |
|-|-|
| id_cliente | 11 dígitos |
| nome | string |
| email | name@example.* |
| regiao | Sul/Sudeste/Centro-Oeste/Nordeste/Norte |
| idade | 2-3 dígitos (max 120) |
| dia_vencimento_fatura | 1-2 dígitos |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| id_cliente duplicado | VERIFICAR NOTAS |
| null em id_cliente/nome | anular |
| e-mail sem @ | anular e-mail |
| regiao inexistente | anular regiao |
| idade > 120 ou idade < 0 ou idade inexistente | anular cliente |
| dia_vencimento_fatura > 31 | anular dia_vencimento_fatura |

**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.
- fonte da verdade dos clientes, precisa validar antes das tabelas que consomem o id_cliente
- vale manter o dado do e-mail, principalmente do provedor, para verificar tendências não óbvias (talvez certos provedores tenham mais chance de dar problema no cadastro, por exemplo)
- se considerarmos ID como CPF, forçar string e anular os que não tiverem 11 dígitos (pra não comer 0 a esquerda)

## BASE_ATENDENTES

| Coluna | Formato |
|-|-|
| id_atendente | 1+ dígitos |
| nome_atendente | String |
| nivel_atendimento | Nível X ou X |
| data_admissao | YYYY-MM-DD |
| data_desligamento | YYYY-MM-DD ou null |
| status_atendente | ativo/inativo |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| null em qualquer entrada fora desligamento | dados essenciais |
| data_admissão > data_desligamento | saiu antes de entrar, anular |
| data_desligamento != null e status ativo | saiu mas está ativo, anular |
| data_desligamento == null e status inativo | está inativo mas não possui data de desligamento, anular |
| tem chamados antes da admissão ou depois do desligamento | anular chamados em questão |


**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.
- Essa é a fonte de verdade dos atendentes, deve ser limpa antes da tabela "chamados"
- Não achei nenhuma inconsistencia de datas (pude ver manualmente, a tabela eh pequena), mas vale deixar as regras explicitas

## BASE_MOTIVOS

| Coluna | Formato |
|-|-|
| id_motivo | 1+ dígitos |
| nome_motivo | String |
| categoria | Cartao/Beneficios/Atendimento/Financeiro/Cadastral/null |
| criticidade | alta/media/baixa (padronizar para caixa baixa) |

**inconsistências**

- ausência de ID ou nome (sem exemplos na tabela), anular

## CANAIS

| Coluna | Formato |
|-|-|
| id_canal | 1 dígito |
| nome_canal | String |
| tipo_canal | Autosservico/Humano |
| status_canal | ativo/inativo |

**inconsistências**

- Ausência de qualquer entrada (sem exemplos na planilha)
- Autosserviço vs. autosservico (sem exmeplos na planilha)

## CHAMADOS

| Coluna | Formato |
|-|-|
| id_chamado | 5+ dígitos |
| id_cliente | 11 dígitos |
| motivo | nome_motivo (de base_motivos) |
| canal | nome_canal (de canais) |
| resolvido | Sim/S/Não/N/1/0/Nao resolvido (padronizar) |
| hora_abertura_chamado | vazio (cruzar com chamados_hora) |
| hora_inicio_atendimento | vazio ou "igual a hora de abertura" |
| hora_finalizacao_atendimento | vazio |
| tempo_espera | "NULL" ou 1-3 dígitos (segundos corridos, validar com chamados_hora) |
| tempo_atendimento | "NULL" ou 1-3 dígitos (segundos corridos, validar com chamados_hora) |
| id_atendente | 1+ dígios |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| null em id_chamado ou id_cliente | dados essenciais |
| resolvido mas não finalizado | cruzar com a tabela com horarios para verificar, anular |
| id_atendente/id_cliente inconsistentes | verificar nas tabelas, anular |
| tempo_espera/tempo_atendimento discorda com a tabela de horas | anular |
| motivo inexistente | anular |
| inconsistência temporal | assumir tempo mais alto |

**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.
- o tempo aqui é medido em segundos corridos

## CHAMADOS_HORA

| Coluna | Formato |
|-|-|
| ID_Chamado | 5+ dígitos |
| ID_Cliente | 11 dígitos |
| Hora_Abertura_Chamado | DD/MM/YYYY às HH:MM:SS |
| Hora_Inicio_Atendimento | DD/MM/YYYY às HH:MM:SS |
| Hora_Finalizacao_Atendimento | DD/MM/YYYY às HH:MM:SS |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| às vs. as | retirar acentos no tratamento inicial |
| Hora_Inicio_Atendimento < Hora_Abertura_Chamado | Chamado atendido antes de ser aberto |
| Hora_Finalizacao_Atendimento < Hora_Inicio_Atendimento | Cuidado com madrugada (começa às 23 e termina 00h) |
| ID_Chamado duplicado | Não usar nenhum dos dois |
| ID_Cliente inválido | Cruzar com clientes.csv |
| ID_Chamado inválido | Cruzar com chamados.csv |
| null em ID | Não dá pra validar |
| null APENAS em Hora_Inicio_Atendimento | Não dá pra saber quanto tempo o atendimento durou |
| null em 2 Hora_* | Não dá pra saber nem quanto tempo levou pra iniciar nem pra terminar |

**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.
- Pode ter null em apenas um Hora_*, desde que não seja o "Hora_Inicio_Atendimento", pois pelo menos tem o dado de quanto tempo levou pra iniciar ou pra terminar.
- Precisa verificar PRIMEIRO a duplicação de ID na tabela referência (clientes/chamados.csv), pra invalidar lá.

## CUSTOS

| Coluna | Formato |
|-|-|
| id_custo | 5+ dígitos |
| id_chamado | 5+ dígitos |
| custo | x,xx ou R$ x,xx ou x,xxreais (usa ponto ou vírgula, considerar sempre como separador de centavos) |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| id_chamado ou id_custo vazio | anular |
| id_custo duplicado | assumir mais caro |
| custo vazio (não 0) | anular |
| custo negativo | anular |
| id_chamado orfão | anular |

**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.

## PESQUISA_SATISFACAO

id_pesquisa	id_chamado	nota_atendimento

| Coluna | Formato |
|-|-|
| id_pesquisa | 5+ dígitos |
| id_chamado | 5+ dígitos |
| nota_atendimento | 1 dígito (0-5) |

**inconsistências**

| Inconsistência | Explicação |
|-|-|
| id_pesquisa vazio | anular |
| id_pesquisa duplicado | anular ambos |
| id_chamado null ou orfão | anular |
| id_chamado duplicado | anular ambos |
| nota null | anular |
| nota negativa | anular |
| nota > 5 | anular |

**NOTAS**
- em caso de id duplicado: 1. se são idênticos, mantêm; 2. se são diferentes, mas concordam (um tem dados a mais que o outro), mescla; 3. se discordam, anule ambos.